# adversary loss の epoch 推移 — adversary head は属性を読めていたか

post-hoc probe では、λ を 10 まで上げても表現から sex / race / age が読める
（[attribute_probe.md](reports/attribute_probe.md)）。GRL が backbone へ返す勾配は adversary head の
予測誤差から来るので、**head 自身が属性を当てられていなければ、backbone は属性を消す方向へ押されない**。

ここでは adversary loss の成分（`attribute_adversary/{sex,race,age}`）を epoch ごとに並べ、
**特徴を見ずに事前分布だけを答える予測器**の loss と比べる。loss がその水準に張り付いていれば、
head は特徴を使っておらず、GRL の勾配もほぼ 0 になっている。

対象は adversary loss を記録するようにした後に ws11 で回した 4 本（λ=0 / 1 / 3 / 10、seed 43）。
seed 1 本なので、読めるのは推移の形であり、λ 間の差の有意性ではない。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET, run_dir, split_csv  # noqa: E402
from analysis.common.run_artifacts import read_config, read_epoch_metrics  # noqa: E402

matplotlib.style.use(STYLE_SHEET)

## 対象 run と定数

λ は run-id から区別できないので、各 run の `config.yaml` の `attribute_adversary_weight` を読んで確かめる。
λ=0 は総損失が `task + 0·adv` なので head に勾配が流れず、head は初期値のまま評価される。
学習された head と比べる相手ではなく、「学習しない head の loss」の参照として置く。

In [ ]:
STUDY = "adversary_strength"
DATASET = "chexpert"
RUN_IDS = {
    0.0: "20260923T075004Z-resnet-chexpert-attribute-invariant-s43-1bae",
    1.0: "20260923T075021Z-resnet-chexpert-attribute-invariant-s43-e41b",
    3.0: "20260923T075026Z-resnet-chexpert-attribute-invariant-s43-ac28",
    10.0: "20260923T075033Z-resnet-chexpert-attribute-invariant-s43-753c",
}
ATTRIBUTES = ["sex", "race", "age"]
LAMBDA_COLOR = {0.0: "#949494", 1.0: "#0173B2", 3.0: "#DE8F05", 10.0: "#029E73"}
# 事前分布との差がこの値を下回った epoch を「張り付いた」とみなす。loss の小数第 3 位の揺れより小さい。
STUCK_TOLERANCE = 0.005

RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"

for weight, run_id in RUN_IDS.items():
    recorded = read_config(run_dir(STUDY, run_id) / "config.yaml")["model"]["loss_fn"]["attribute_adversary_weight"]
    assert float(recorded) == weight, (run_id, recorded)

## 事前分布だけを答える予測器の loss

adversary の loss は、属性が観測された行だけで計算される（`objectives/attribute_invariance.py`）。
参照も同じく `*_missing` が偽の行だけで作る。

- sex / race（cross-entropy）: train の観測行の頻度を答え続けたときの loss。train では周辺分布のエントロピーになる。
- age（MSE）: 学習時は train の平均と標準偏差で標準化されるので、常に 0（train 平均）を答えたときの MSE。
  train では 1 になり、val では val の分散と平均のずれの分だけ 1 からずれる。

race の観測行は train の 89% で、全行で計算したエントロピー（1.010）より高くなる点に注意する。

In [ ]:
def prior_losses(train: pd.DataFrame, evaluated: pd.DataFrame) -> dict[str, float]:
    """train の頻度・平均だけを答える予測器の loss を、evaluated の観測行で計算する。

    Args:
        train: 頻度と標準化の統計量を取る split
        evaluated: loss を評価する split

    Returns:
        dict[str, float]: 属性名ごとの loss
    """
    losses = {}
    for name in ["sex", "race"]:
        train_observed = train.loc[~train[f"{name}_missing"].astype(bool), name]
        evaluated_observed = evaluated.loc[~evaluated[f"{name}_missing"].astype(bool), name]
        prior = train_observed.value_counts(normalize=True)
        losses[name] = float(-np.log(evaluated_observed.map(prior)).mean())
    train_age = train.loc[~train["age_missing"].astype(bool), "age"]
    evaluated_age = evaluated.loc[~evaluated["age_missing"].astype(bool), "age"]
    standardized = (evaluated_age - train_age.mean()) / train_age.std()
    losses["age"] = float((standardized**2).mean())
    return losses


train_split = pd.read_csv(split_csv(DATASET, "train"))
val_split = pd.read_csv(split_csv(DATASET, "val"))
prior = pd.DataFrame({"train": prior_losses(train_split, train_split), "val": prior_losses(train_split, val_split)}).T
prior.round(4)

## epoch ごとの adversary loss

`metrics/metrics.csv` は train と val を別行に書くので、`read_epoch_metrics` で epoch ごとの 1 行に束ねる。

In [ ]:
epoch_rows = []
for weight, run_id in RUN_IDS.items():
    for row in read_epoch_metrics(run_dir(STUDY, run_id) / "metrics" / "metrics.csv"):
        epoch_rows.append({"lambda": weight, "run_id": run_id, **row})
epochs = pd.DataFrame(epoch_rows)

loss_columns = [f"{split}/attribute_adversary/{name}" for split in ["train", "val"] for name in ATTRIBUTES]
epoch_losses = epochs[["lambda", "run_id", "epoch", *loss_columns, "val/auroc"]]
epoch_losses.to_csv(RESULTS / "adversary_loss_epochs.csv", index=False)
epoch_losses.pivot_table(index="epoch", columns="lambda", values="val/attribute_adversary/sex").round(4)

## 推移の図

点線が事前分布だけを答える予測器の val loss。学習された head（λ≥1）がこの線に重なっていれば、
head は特徴を使っていない。

In [ ]:
figure, axes = plt.subplots(1, len(ATTRIBUTES), figsize=(12, 3.6), sharex=True)
for axis, name in zip(axes, ATTRIBUTES, strict=True):
    for weight, group in epoch_losses.groupby("lambda"):
        axis.plot(
            group["epoch"],
            group[f"val/attribute_adversary/{name}"],
            marker="o",
            markersize=3,
            color=LAMBDA_COLOR[weight],
            label=f"λ={weight:g}",
        )
    axis.axhline(prior.loc["val", name], color="#000000", linestyle=":", linewidth=1.2, label="prior only")
    axis.set_title(name)
    axis.set_xlabel("epoch")
axes[0].set_ylabel("val adversary loss")
# λ=0 の race は初期化のままで 1.8 付近にあり、そのままでは λ≥1 の差が潰れるので縦軸を絞る。
axes[1].set_ylim(prior.loc["val", "race"] - 0.05, 1.5)
axes[-1].legend(frameon=False, fontsize=8)
figure.tight_layout()
figure.savefig(FIGURES / "adversary_loss_epochs_val.png", dpi=200)
plt.show()

## 事前分布との差と、張り付いた epoch

最終 epoch の loss から事前分布の loss を引いた値（負なら事前分布より良い）と、
その差が `STUCK_TOLERANCE` 未満になり、以後ずっとそのままだった最初の epoch を出す。

In [ ]:
def first_stuck_epoch(frame: pd.DataFrame, column: str, reference: float) -> float:
    """差が許容幅に入り、最終 epoch まで出なかった最初の epoch を返す。入らなければ NaN。

    Args:
        frame: 1 run の epoch 昇順の行
        column: loss の列名
        reference: 事前分布の loss

    Returns:
        float: epoch 番号
    """
    within = (frame[column] - reference).abs() < STUCK_TOLERANCE
    if not within.iloc[-1]:
        return float("nan")
    leaving = within[~within]
    return float(frame["epoch"].min() if leaving.empty else frame.loc[leaving.index[-1] + 1, "epoch"])


summary_rows = []
for weight, group in epoch_losses.groupby("lambda"):
    group = group.sort_values("epoch").reset_index(drop=True)
    last = group.iloc[-1]
    for name in ATTRIBUTES:
        column = f"val/attribute_adversary/{name}"
        summary_rows.append(
            {
                "lambda": weight,
                "attribute": name,
                "final_val_loss": last[column],
                "prior_val_loss": prior.loc["val", name],
                "final_minus_prior": last[column] - prior.loc["val", name],
                "min_val_loss": group[column].min(),
                "stuck_from_epoch": first_stuck_epoch(group, column, prior.loc["val", name]),
            }
        )
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS / "adversary_loss_vs_prior.csv", index=False)
summary.round(4)

## まとめ

- λ=1 / 3 / 10 の sex の adversary loss は、それぞれ epoch 4 / 1 / 3 以降、事前分布だけを答える予測器の val loss（0.6737）との差が 0.005 未満に収まり、最終 epoch まで出ない。最終 epoch の差は 0.0002〜0.0008。
- race も epoch 21〜25 以降は同じ状態になる（最終 epoch の差 0.0005〜0.0023、事前分布は 1.0786）。
- age は epoch ごとに揺れ、事前分布（0.981）をわずかに下回る。最終 epoch の差は λ=1 で −0.023、λ=3 / 10 で −0.006〜−0.007。標準化した age の R² に直すと 0.01〜0.05 程度で、ほぼ読めていない。
- λ を 1 から 10 に上げても、張り付く水準は変わらない。
- λ=0 は head が学習されないので、sex 0.693（2 値の一様分布）と race 1.84（初期化のまま）に留まる。

**解釈候補**: head が事前分布だけを答えるなら、GRL が backbone へ返す勾配は、属性を消す方向の成分をほとんど持たない。
epoch の早い段階からは、λ によらず実質的に adversary の無い学習になっている可能性がある。
head の中で何が起きているかは [adversary_head_state.ipynb](adversary_head_state.ipynb) で見る。